## 1 · Environment Setup

In [1]:
!pip install -q torch torchvision pandas pillow tqdm

In [2]:
import os
import re
import math
import random
import string
import pickle
from collections import Counter
from PIL import Image

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Using device: cpu


## 2 · Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR      = '/content/drive/MyDrive/caption_data'
SAVE_DIR      = os.path.join(BASE_DIR, 'trained')
os.makedirs(SAVE_DIR, exist_ok=True)

# Copy data to local disk for faster I/O
print("Copying data to local disk...")
!cp -r /content/drive/MyDrive/caption_data/Images /content/Images
!cp /content/drive/MyDrive/caption_data/captions.txt /content/captions.txt
print("Done.")

CAPTIONS_FILE = '/content/captions.txt'
IMAGES_DIR    = '/content/Images'

print(f'  captions : {CAPTIONS_FILE}')
print(f'  images   : {IMAGES_DIR}')
print(f'  save_dir : {SAVE_DIR}')

Mounted at /content/drive
Copying data to local disk...
Done.
  captions : /content/captions.txt
  images   : /content/Images
  save_dir : /content/drive/MyDrive/caption_data/trained


## 3 · Hyperparameters

In [4]:
MIN_WORD_FREQ = 5       # words appearing fewer times are mapped to <UNK>
TRAIN_SPLIT   = 0.80    # fraction of *images* used for training
IMG_SIZE      = 224

EMBED_DIM   = 256
HIDDEN_DIM  = 512
NUM_LAYERS  = 1
DROPOUT     = 0.5

BATCH_SIZE  = 64
NUM_EPOCHS  = 30
LR          = 3e-4
GRAD_CLIP   = 5.0
LR_PATIENCE = 3        # epochs of no val improvement before halving LR
LR_FACTOR   = 0.5

PAD_TOKEN   = '<PAD>'
UNK_TOKEN   = '<UNK>'
START_TOKEN = '<START>'
END_TOKEN   = '<END>'

print('Hyperparameters loaded.')

Hyperparameters loaded.


## 4 · Data Loading & Text Cleaning

In [5]:
def load_captions(filepath: str) -> pd.DataFrame:
    """
    Read captions.txt.
    Expected format (Flickr8k style): each line is  'image_file,caption'
    or tab-separated.  The first column is the image filename.
    """
    rows = []
    with open(filepath, 'r', encoding='utf-8') as fh:
        for i, line in enumerate(fh):
            line = line.strip()
            if not line or line.lower().startswith('image'):
                continue  # skip header or blank lines
            # Support comma or tab as separator
            if '\t' in line:
                parts = line.split('\t', 1)
            else:
                parts = line.split(',', 1)
            if len(parts) != 2:
                continue
            image_name, caption = parts[0].strip(), parts[1].strip()
            # Some Flickr8k files append '#N' to the filename  (e.g. dog.jpg#0)
            image_name = image_name.split('#')[0]
            rows.append({'image': image_name, 'caption': caption})
    df = pd.DataFrame(rows)
    print(f'Loaded {len(df):,} caption rows for {df["image"].nunique():,} unique images.')
    return df


df_raw = load_captions(CAPTIONS_FILE)
df_raw.head()

Loaded 40,455 caption rows for 8,091 unique images.


,image,caption
0,1000268201_693b08cb0e.jpg,A child in a pink dress is climbing up a set o...
1,1000268201_693b08cb0e.jpg,A girl going into a wooden building .
2,1000268201_693b08cb0e.jpg,A little girl climbing into a wooden playhouse .
3,1000268201_693b08cb0e.jpg,A little girl climbing the stairs to her playh...
4,1000268201_693b08cb0e.jpg,A little girl in a pink dress going into a woo...


In [6]:
ARTICLES = {'a', 'an', 'the'}


def clean_caption(text: str) -> str:
    """Lowercase, remove punctuation & digits, strip articles and 1-char words."""
    text = text.lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Remove digits
    text = re.sub(r'\d+', '', text)
    # Tokenise on whitespace
    tokens = text.split()
    # Drop single-character tokens and articles
    tokens = [t for t in tokens if len(t) > 1 and t not in ARTICLES]
    # Wrap with START / END
    tokens = [START_TOKEN] + tokens + [END_TOKEN]
    return ' '.join(tokens)


df = df_raw.copy()
df['caption'] = df['caption'].apply(clean_caption)

print('Sample cleaned captions:')
for _, row in df.head(3).iterrows():
    print(f'  {row["image"]}  →  {row["caption"]}')

Sample cleaned captions:
  1000268201_693b08cb0e.jpg  →  <START> child in pink dress is climbing up set of stairs in entry way <END>
  1000268201_693b08cb0e.jpg  →  <START> girl going into wooden building <END>
  1000268201_693b08cb0e.jpg  →  <START> little girl climbing into wooden playhouse <END>


## 5 · Vocabulary

In [7]:
class Vocabulary:
    """Word ↔ index mapping with frequency-based filtering."""

    def __init__(self, min_freq: int = 5):
        self.min_freq = min_freq
        # Reserve indices for special tokens
        self.word2idx = {
            PAD_TOKEN: 0,
            UNK_TOKEN: 1,
            START_TOKEN: 2,
            END_TOKEN: 3,
        }
        self.idx2word = {v: k for k, v in self.word2idx.items()}
        self._counter: Counter = Counter()

    # Special token indices (properties for convenience)
    @property
    def pad_idx(self):   return self.word2idx[PAD_TOKEN]
    @property
    def unk_idx(self):   return self.word2idx[UNK_TOKEN]
    @property
    def start_idx(self): return self.word2idx[START_TOKEN]
    @property
    def end_idx(self):   return self.word2idx[END_TOKEN]

    def __len__(self):
        return len(self.word2idx)

    def build(self, captions: pd.Series) -> None:
        """Count all words in *captions* and add those above min_freq."""
        for caption in captions:
            for word in caption.split():
                if word not in (PAD_TOKEN, UNK_TOKEN, START_TOKEN, END_TOKEN):
                    self._counter[word] += 1
        for word, freq in self._counter.items():
            if freq >= self.min_freq and word not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx]  = word
        print(f'Vocabulary built: {len(self):,} tokens  '
              f'(min_freq={self.min_freq}, '
              f'rare_words_collapsed_to_UNK: '
              f'{sum(1 for f in self._counter.values() if f < self.min_freq):,})')

    def numericalize(self, sentence: str) -> list[int]:
        """Convert a whitespace-tokenised string to a list of indices."""
        return [
            self.word2idx.get(word, self.unk_idx)
            for word in sentence.split()
        ]

    def decode(self, indices: list[int], skip_special: bool = True) -> str:
        """Convert indices back to a human-readable string."""
        special = {self.pad_idx, self.start_idx, self.end_idx}
        words = []
        for idx in indices:
            if skip_special and idx in special:
                if idx == self.end_idx:
                    break
                continue
            words.append(self.idx2word.get(idx, UNK_TOKEN))
        return ' '.join(words)


# Build vocabulary from ALL captions (train + val share the same vocab)
vocab = Vocabulary(min_freq=MIN_WORD_FREQ)
vocab.build(df['caption'])

print(f'\nVocabulary size : {len(vocab)}')
print(f'PAD idx         : {vocab.pad_idx}')
print(f'START idx       : {vocab.start_idx}')
print(f'END idx         : {vocab.end_idx}')

Vocabulary built: 2,983 tokens  (min_freq=5, rare_words_collapsed_to_UNK: 5,784)

Vocabulary size : 2983
PAD idx         : 0
START idx       : 2
END idx         : 3


## 6 · Train / Validation Split (by Image)

In [8]:
all_images = df['image'].unique().tolist()
random.shuffle(all_images)

n_train    = int(len(all_images) * TRAIN_SPLIT)
train_imgs = set(all_images[:n_train])
val_imgs   = set(all_images[n_train:])

df_train = df[df['image'].isin(train_imgs)].reset_index(drop=True)
df_val   = df[df['image'].isin(val_imgs)].reset_index(drop=True)

print(f'Train: {len(train_imgs):,} images  ({len(df_train):,} captions)')
print(f'Val  : {len(val_imgs):,} images  ({len(df_val):,} captions)')

# Sanity-check: no image appears in both splits
assert train_imgs.isdisjoint(val_imgs), 'Leak detected!'
print('No image leakage between splits. ✓')

Train: 6,472 images  (32,360 captions)
Val  : 1,619 images  (8,095 captions)
No image leakage between splits. ✓


## 7 · Dataset & DataLoader

In [9]:
# ImageNet normalisation
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print('Transforms defined.')

Transforms defined.


In [10]:
class CaptionDataset(Dataset):
    """
    Returns (image_tensor, caption_indices) pairs.
    Each (image, caption) row in the DataFrame becomes one sample.
    """

    def __init__(
        self,
        dataframe: pd.DataFrame,
        images_dir: str,
        vocabulary: Vocabulary,
        transform=None,
    ):
        self.df         = dataframe.reset_index(drop=True)
        self.images_dir = images_dir
        self.vocab      = vocabulary
        self.transform  = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row     = self.df.iloc[idx]
        img_path = os.path.join(self.images_dir, row['image'])

        # Load image
        try:
            image = Image.open(img_path).convert('RGB')
        except (FileNotFoundError, OSError):
            # Return a blank image if file is missing (graceful degradation)
            image = Image.new('RGB', (IMG_SIZE, IMG_SIZE))

        if self.transform:
            image = self.transform(image)

        # Numericalize caption
        caption_ids = torch.tensor(
            self.vocab.numericalize(row['caption']),
            dtype=torch.long,
        )
        return image, caption_ids



def collate_fn(batch, pad_idx: int):
    """
    Pads variable-length caption sequences to the length of the longest
    caption in the batch.  Returns (images, captions) tensors.
    """
    images, captions = zip(*batch)

    # Stack images  →  (B, C, H, W)
    images = torch.stack(images, dim=0)

    # Pad captions  →  (B, T_max)
    captions = torch.nn.utils.rnn.pad_sequence(
        captions,
        batch_first=True,
        padding_value=pad_idx,
    )
    return images, captions


import functools

train_dataset = CaptionDataset(df_train, IMAGES_DIR, vocab, train_transform)
val_dataset   = CaptionDataset(df_val,   IMAGES_DIR, vocab, val_transform)

_collate = functools.partial(collate_fn, pad_idx=vocab.pad_idx)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=(DEVICE.type == 'cuda'),
    collate_fn=_collate,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=(DEVICE.type == 'cuda'),
    collate_fn=_collate,
)

print(f'Train batches : {len(train_loader):,}')
print(f'Val batches   : {len(val_loader):,}')

sample_imgs, sample_caps = next(iter(train_loader))
print(f'\nSample batch — images: {tuple(sample_imgs.shape)}, '
      f'captions: {tuple(sample_caps.shape)}')

Train batches : 506
Val batches   : 127


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()



Sample batch — images: (64, 3, 224, 224), captions: (64, 17)
